In [1]:
import json
import sys
import os
import time
from tqdm import tqdm

from openai import OpenAI
from dotenv import load_dotenv


from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage


# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)

In [2]:
# function for getting embeddings from API
def get_embedding(principle):
    num_retries = 0
    max_retries = 5
    while True:
        try:
            response = client.embeddings.create(
                input=principle,
                model="text-embedding-3-small"
            )
            return response.data[0].embedding
        except Exception as e:
            print(f"Error: {e}. Retrying in 5 seconds...")
            num_retries += 1
            if num_retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            time.sleep(5)


In [ ]:
# Start with this block for contextual principles format

with open(r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\ConsensusSelection\consensus_principle_selection_th=0.42_complete_linkage.json', 'r',encoding='utf-8') as f:
    scored_principles = json.load(f)

print(scored_principles[0])


clustering_tuples = []
for item in tqdm(scored_principles):
    principle = item['summary_text']
    embedding = get_embedding(principle)
    score = item['global_msd']
    clustering_tuples.append((principle, embedding, score))


{'rank': 1, 'cluster_id': 62, 'summary_text': 'AI should be honest and truthful.', 'global_msd': 0.0004292603555339397}


100%|██████████| 276/276 [02:19<00:00,  1.99it/s]


In [3]:
# Start with this block for decontextual principles format, everything after should be the same

with open(r'C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\FairClustering\cluster_representatives_greedy_capture_k=1371.json', 'r') as f:
    fair_clusters= json.load(f)

print(fair_clusters[0])

clustering_tuples = []
for item in tqdm(fair_clusters):
    principle = item['representative_principle']
    embedding = get_embedding(principle)
    score = None
    clustering_tuples.append((principle, embedding, score))


{'cluster_id': 0, 'df_index': 211, 'representative_principle': 'AI should communicate respectfully.'}


100%|██████████████████████████████████████████████████████████████████████████████| 1129/1129 [05:28<00:00,  3.43it/s]


In [7]:
# cluster 
embeddings = []
for item in clustering_tuples:
    embeddings.append(item[1])


clustering_model = AgglomerativeClustering(n_clusters=None, linkage = 'complete', metric = 'cosine', distance_threshold=.3)  # Adjust distance_threshold as needed
labels = clustering_model.fit_predict(embeddings)


In [8]:
#print number of clusters
num_clusters = len(set(labels))
print(f"Number of clusters: {num_clusters}")

#save clusters & principles to json
clusters = {}
for i, label in enumerate(labels):
    if label not in clusters:
        clusters[label] = []
    clusters[label].append((clustering_tuples[i][0], clustering_tuples[i][2]))

clusters_output = []

for i in range(len(clusters)):
    clusters_output.append({
        "cluster_id": i,
        "principles": clusters[i]
    })

with open('intermediate_principle_clusters_k=1371_json', 'w') as f:
    json.dump(clusters_output, f, indent=4)

Number of clusters: 130


In [9]:
print(clusters_output[0])

# Get highest scoring principle per cluster

deduped_principles = []

for cluster in clusters_output:
    highest_scorer = min(cluster['principles'], key=lambda x: x[1])

    deduped_principles.append(highest_scorer)
    

deduped_principles.sort(key=lambda x: x[1])

print(deduped_principles)

with open('deduped_12_23_decontextual_k=1371.json', 'w') as f:
    json.dump(deduped_principles, f, indent=4)

{'cluster_id': 0, 'principles': [('AI should prioritize ethical behavior in all contexts.', None), ('AI should behave ethically.', None), ('AI should behave ethically and without bias.', None), ('AI should be respectful towards all cultures and every individual.', None), ('AI should consistently exhibit respect for all human cultures.', None), ('AI must respect ethical considerations.', None), ('AI should prioritize ethicality in all contexts.', None), ('AI should demonstrate consideration for people and cultures.', None), ('AI should consistently prioritize ethical and responsible behavior.', None)]}


TypeError: '<' not supported between instances of 'NoneType' and 'NoneType'